# PCG-MAS · run the 2 large-LLM cells on Colab Pro+ (A100 / H100)

This notebook runs the experiments that **cannot** fit on a MacBook:
- `Llama-3.3-70B` — 70B params, needs A100 80 GB or H100 with 4-bit quant
- `deepseek-v3`   — 671B MoE, called via HF Inference API (no local weights)

The other 5 LLMs (phi-3.5-mini, qwen2.5-7B, deepseek-llm-7b-chat, Llama-3.1-8B, Gemma-2-9b-it) are run locally on the MacBook — see the project README for that side.

## Workflow
1. Mount Drive (project lives at `/content/drive/MyDrive/proof-carrying-multi-agents`)
2. Install deps into `/usr/local` directly (do NOT use a Drive-side venv — much too slow)
3. Set HF cache to a 5 TB Drive folder so model weights download once, not per session
4. Verify GPU
5. Run all 5 R-experiments × 2 large LLMs × 8 datasets, writing per-cell JSONs into `results/{r_id}-{llm}-{dataset}-...` so `pick_top_k.py` picks them up later

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/proof-carrying-multi-agents'
!ls "$PROJECT" | head

## Cell 2 — Clean PATH and confirm system Python

Your previous notebook had `/content/multi-agents/bin` lingering on PATH after `rm -rf`.  We strip it explicitly so `!python` resolves to the Colab system Python (3.11).

In [ ]:
import os, sys

bad = '/content/multi-agents/bin'
os.environ['PATH'] = ':'.join(p for p in os.environ['PATH'].split(':') if p != bad)
!rm -rf /content/multi-agents

!which python && python --version
print('sys.executable =', sys.executable)

## Cell 3 — Configure HF cache on Drive (one-time download per LLM)

Llama-3.3-70B is ~140 GB in FP16 weights, ~40 GB in 4-bit. We point HF_HOME at a Drive folder so the download survives runtime resets — your 5 TB Drive can hold all the LLMs comfortably.

In [ ]:
import os

DRIVE_HF_CACHE = '/content/drive/MyDrive/hf_cache'
os.makedirs(DRIVE_HF_CACHE, exist_ok=True)
os.environ['HF_HOME']           = DRIVE_HF_CACHE
os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_HF_CACHE}/datasets'
os.environ['TRANSFORMERS_CACHE']= f'{DRIVE_HF_CACHE}/transformers'

# YOUR HF TOKEN — must have read access to gated Llama-3.3-70B repo.
# Llama-3.3-70B is a gated model: visit
#     https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct
# and accept the license under your account FIRST. Then paste the token below.
os.environ['HF_TOKEN'] = 'hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

print('HF cache  :', os.environ['HF_HOME'])
print('HF token  :', 'set' if os.environ['HF_TOKEN'].startswith('hf_') else 'NOT SET — fix above')

## Cell 4 — Install deps into the Colab system Python

Two notes:
1. We DO NOT create a venv on Drive — Drive I/O is brutally slow for `pip install`. Install into the Colab Python directly.
2. `bitsandbytes` is required for 4-bit Llama-70B loading; flash-attn speeds inference but is optional.

In [ ]:
%cd $PROJECT

!pip install -q -e .
!pip install -q transformers>=4.46 accelerate>=1.0 datasets>=3.0 \
    bitsandbytes>=0.44 sentencepiece protobuf \
    huggingface_hub[hf_xet]

# flash-attn is optional but cuts inference time by ~30% on H100.
# Comment out if it fails to build; the rest of the pipeline doesn't need it.
!pip install -q flash-attn --no-build-isolation || echo '(flash-attn skipped, not critical)'

## Cell 5 — Verify GPU

Llama-3.3-70B in 4-bit needs ≥40 GB VRAM. On Colab Pro+:
- A100 40 GB: works with 4-bit
- A100 80 GB: works with 4-bit comfortably
- H100 80 GB: works with 4-bit and is fastest

If you see a T4 or V100 here, change Runtime → Runtime type → GPU type.

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
print('Device         :', torch.cuda.get_device_name(0))
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM           : {vram_gb:.1f} GB')

if vram_gb < 38:
    raise RuntimeError(
        f'GPU has only {vram_gb:.1f} GB VRAM. Llama-3.3-70B in 4-bit '
        f'needs ~40 GB. Switch to A100 or H100 in Runtime settings.'
    )

## Cell 6 — Sanity: load Llama-3.3-70B once and confirm it works

First-time load downloads ~140 GB of FP16 weights to your Drive cache (one-shot, ~10-15 min on a Colab Pro connection). Subsequent sessions reuse the cache, taking ~3-5 min just to load weights into VRAM.

We use `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=bfloat16)` to fit on a 40-80 GB GPU.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL = 'meta-llama/Llama-3.3-70B-Instruct'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tok = AutoTokenizer.from_pretrained(MODEL)
mdl = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='sdpa',  # change to 'flash_attention_2' if flash-attn installed
)
print('Llama-3.3-70B loaded. VRAM after load:',
      f'{torch.cuda.memory_allocated() / 1e9:.1f} GB')

# Quick smoke generation
msgs = [{'role':'user','content':'Reply with exactly the word: ready'}]
inp = tok.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to('cuda')
with torch.inference_mode():
    out = mdl.generate(inp, max_new_tokens=8, do_sample=False)
print('Sanity output:', tok.decode(out[0][inp.shape[1]:], skip_special_tokens=True))

# Free up before the run (each R-script will reload as needed)
del mdl, tok
torch.cuda.empty_cache()

## Cell 7 — Run R1..R5 with Llama-3.3-70B across 8 datasets

We pass `--track-cell <llm>:<dataset>` so each run writes a `cell.json` file that `pick_top_k.py` picks up later.

**Wall time on H100**: ~2-3 hours per (R-experiment × dataset) combo at n=200 examples. Total: 5 experiments × 8 datasets × 2-3h = roughly 80-120 H100-hours per LLM.

If your Colab budget is tight, reduce N_EXAMPLES to 100 and start with the 4 most diverse datasets (HotpotQA, TAT-QA, WebLINX, FEVER).

In [ ]:
import subprocess, time
from pathlib import Path

PROJECT = '/content/drive/MyDrive/proof-carrying-multi-agents'
LLM = 'meta-llama/Llama-3.3-70B-Instruct'
LLM_SHORT = 'Llama-3.3-70B'
N_EXAMPLES = 200
SEEDS = '0 1 2'

# 8 datasets × 5 R-experiments
DATASETS = ['hotpotqa', 'twowiki', 'toolbench', 'fever',
            'pubmedqa', 'tatqa', 'weblinx', 'synthetic']
EXPERIMENTS = [
    ('r1', 'run_r1_checkability.py'),
    ('r2', 'run_r2_redundancy.py'),
    ('r3', 'run_r3_responsibility.py'),
    ('r4', 'run_r4_risk_privacy.py'),
    ('r5', 'run_r5_overhead.py'),
]

for r_id, script in EXPERIMENTS:
    for ds in DATASETS:
        cfg = f'configs/{r_id}_{ds}.yaml'
        if not Path(f'{PROJECT}/{cfg}').exists():
            # Some R-experiments share a single config (e.g. r2_redundancy.yaml).
            # Use the canonical one and pass --dataset on the command line.
            cfg = f'configs/{r_id}_redundancy.yaml' if r_id == 'r2' else \
                  f'configs/{r_id}_responsibility.yaml' if r_id == 'r3' else \
                  f'configs/{r_id}_risk.yaml' if r_id == 'r4' else \
                  f'configs/{r_id}_overhead.yaml' if r_id == 'r5' else \
                  f'configs/{r_id}_hotpotqa.yaml'
        cmd = (
            f'cd {PROJECT} && python scripts/{script} '
            f'--config {cfg} '
            f'--seeds {SEEDS} '
            f'--n-examples {N_EXAMPLES} '
            f'--backend hf_local '
            f'--model-name {LLM} '
            f'--dataset-override {ds} '
            f'--track-cell {LLM_SHORT}:{ds} '
            f'2>&1 | tail -30'
        )
        print(f'\n>>> {r_id} · {LLM_SHORT} · {ds}')
        t0 = time.time()
        get_ipython().system(cmd)
        elapsed = time.time() - t0
        print(f'    completed in {elapsed/60:.1f} min')

        # Force a Drive flush so the cell.json is durable before the next run
        !sync && sleep 2

## Cell 8 — Run R1..R5 with deepseek-v3 via HF Inference API

deepseek-v3 is a 671B MoE — too large to load on any single GPU. We call it via the HF Inference API (`backend=hf_inference`) instead of `hf_local`. This requires:
- A free HF account token (the one you set in Cell 3)
- DeepSeek-V3 access on HF (it's ungated — no manual approval needed)

Each call costs you nothing on the free tier but is rate-limited (~20 req/min on free, much higher with HF Pro). For 5 experiments × 8 datasets × 200 examples × 3 seeds = 24,000 calls. At 20 req/min you're looking at ~20 hours sustained; with HF Pro ($9/month) it's ~2-3 hours.

In [ ]:
LLM = 'deepseek-ai/DeepSeek-V3'
LLM_SHORT = 'deepseek-v3'
N_EXAMPLES = 200
SEEDS = '0 1 2'

for r_id, script in EXPERIMENTS:
    for ds in DATASETS:
        cfg = f'configs/{r_id}_{ds}.yaml'
        if not Path(f'{PROJECT}/{cfg}').exists():
            cfg = f'configs/{r_id}_redundancy.yaml' if r_id == 'r2' else \
                  f'configs/{r_id}_responsibility.yaml' if r_id == 'r3' else \
                  f'configs/{r_id}_risk.yaml' if r_id == 'r4' else \
                  f'configs/{r_id}_overhead.yaml' if r_id == 'r5' else \
                  f'configs/{r_id}_hotpotqa.yaml'
        cmd = (
            f'cd {PROJECT} && python scripts/{script} '
            f'--config {cfg} '
            f'--seeds {SEEDS} '
            f'--n-examples {N_EXAMPLES} '
            f'--backend hf_inference '
            f'--model-name {LLM} '
            f'--dataset-override {ds} '
            f'--track-cell {LLM_SHORT}:{ds} '
            f'2>&1 | tail -30'
        )
        print(f'\n>>> {r_id} · {LLM_SHORT} · {ds}')
        get_ipython().system(cmd)
        !sync && sleep 2

## Cell 9 — Verify all cell.json files are durable on Drive

After both runs finish, count what we have so the local Mac can pull only what's needed.

In [ ]:
!cd $PROJECT && find results -name 'cell.json' | wc -l
!cd $PROJECT && find results -name 'cell.json' | head -20
!cd $PROJECT && du -sh results/

## Cell 10 — Cleanup before disconnecting

Free GPU + flush all Drive writes.

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
!sync
print('OK to disconnect runtime now.')